> **기록 표시 (2026-11-05):** 이 노트북은 당시 실험 기록이다. 아래 셀 2의 `delete_test_blocks()`는 지금 `chunk_ids` 인자를 받으므로 그대로 다시 실행하면 실패한다. 재실행은 노트북 19를 사용한다.


# 18. 전문 검색(chunk_text) 테스트 — 질문 단어로 순위가 바뀌는지 확인

- 목적: 로컬 키워드 점수(질문과 무관) 대신 수업 38-1의 `chunk_text` 전문 인덱스로 찾았을 때 순위를 확인한다.
- 정책: 공동 DB에는 `PLTest` 라벨과 `test_` 접두사 노드만 넣는다. 끝나면 같은 조건으로 지우고 0개를 확인한다.
- 충돌: 넣기 전에 `test_` ID 충돌을 확인한다 (retrieve.load_test_blocks 안에서 검사).
- 대상: 홍역 지침, 진단·신고기준, 법, 수족구 블록 (허용 문서 4개).

In [1]:
# 1) 준비: 허용 문서와 테스트 질문 정의 (질문을 바꾸지 않는다, 평가 독립성)
import sys
sys.path.insert(0, "../../src")
from rag.prevention_legal.retrieve import load_test_blocks, delete_test_blocks, fulltext_search, lucene_query

DOCS_ALLOWED = ["kdca_measles_2026", "kdca_diagnosis_reporting_2026",
                "moleg_infectious_disease_act_21521", "kdca_hfmd_2026"]
QUESTIONS = [
    "교내 MMR 2차 접종 미완료자는 어떻게 해야 하나요",   # Q05 (새 카드)
    "홍역은 어떻게 전파되나요?",                       # 무관 질문
    "MMR 2차 접종의 부작용은?",                        # 무관 질문
    "홍역 의심환자가 생겼을 때 학교 담당교사와 의료기관은 각각 보건소에 무엇을 신고하거나 안내하나요? 법률과 지침의 조건을 구분해 주세요.",  # 샘플
]
print(lucene_query(QUESTIONS[0]))

"교내" OR "MMR" OR "2차" OR "접종" OR "미완료자는" OR "어떻게" OR "해야" OR "하나요"


In [2]:
# 2) 테스트 노드 넣기 (try 안에서 넣고, finally 에서 반드시 지운다)
from rag.prevention_legal.retrieve import load_test_blocks, delete_test_blocks
try:
    print("넣은 개수:", load_test_blocks(set(DOCS_ALLOWED)))
    # 3) 질문별 상위 10개 결과 확인
    for q in QUESTIONS:
        hits = fulltext_search(q, DOCS_ALLOWED, k=10)
        print("\n질문:", q[:40], "..." if len(q) > 40 else "")
        for i, h in enumerate(hits, 1):
            print(f"  {i:2d} {h['score']:.2f} {h['chunk_id']}")
finally:
    print("\n삭제한 개수:", delete_test_blocks())


넣은 개수: 31


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.fulltext.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('chunk_text', $q) YIELD node, score\n                    WHERE node:PLTest AND node.document_id IN $docs\n                    RETURN node.chunk_id AS chunk_id, node.text AS text, node.document_id AS document_id,\n                           node.pdf_page AS pdf_page, score\n                    ORDER BY scor


질문: 교내 MMR 2차 접종 미완료자는 어떻게 해야 하나요 
   1 9.93 kdca_measles_2026:p0093:b04
   2 2.65 kdca_measles_2026:p0093:b01
   3 0.95 kdca_measles_2026:p0030:t01r01
   4 0.95 kdca_measles_2026:p0030:t01r02
   5 0.91 kdca_measles_2026:p0030:t01r03
   6 0.87 moleg_infectious_disease_act_21521:p0018:b01


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.fulltext.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('chunk_text', $q) YIELD node, score\n                    WHERE node:PLTest AND node.document_id IN $docs\n                    RETURN node.chunk_id AS chunk_id, node.text AS text, node.document_id AS document_id,\n                           node.pdf_page AS pdf_page, score\n                    ORDER BY scor


질문: 홍역은 어떻게 전파되나요? 


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.fulltext.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('chunk_text', $q) YIELD node, score\n                    WHERE node:PLTest AND node.document_id IN $docs\n                    RETURN node.chunk_id AS chunk_id, node.text AS text, node.document_id AS document_id,\n                           node.pdf_page AS pdf_page, score\n                    ORDER BY scor


질문: MMR 2차 접종의 부작용은? 
   1 5.61 kdca_measles_2026:p0093:b04


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.fulltext.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('chunk_text', $q) YIELD node, score\n                    WHERE node:PLTest AND node.document_id IN $docs\n                    RETURN node.chunk_id AS chunk_id, node.text AS text, node.document_id AS document_id,\n                           node.pdf_page AS pdf_page, score\n                    ORDER BY scor


질문: 홍역 의심환자가 생겼을 때 학교 담당교사와 의료기관은 각각 보건소에 무엇 ...
   1 7.23 kdca_measles_2026:p0093:b01
   2 2.59 moleg_infectious_disease_act_21521:p0019:b03
   3 2.29 kdca_diagnosis_reporting_2026:p0089:t01r02
   4 2.11 kdca_measles_2026:p0023:t01r02
   5 2.11 kdca_diagnosis_reporting_2026:p0089:t01r03
   6 2.10 kdca_diagnosis_reporting_2026:p0089:t01r01
   7 2.08 kdca_measles_2026:p0023:t01r01
   8 2.00 kdca_measles_2026:p0093:b03
   9 1.92 kdca_measles_2026:p0093:b04
  10 1.71 kdca_diagnosis_reporting_2026:p0089:t01r04



삭제한 개수: 31


In [3]:
# 4) 삭제 확인: PLTest 노드와 test_ 노드가 0개여야 한다
from neo4j import GraphDatabase
from rag.prevention_legal.neo4j_conn import connect
d = connect()
with d.session() as s:
    print("PLTest:", s.run("MATCH (c:PLTest) RETURN count(c) AS n").single()["n"])
    print("test_ ID:", s.run("MATCH (c:Chunk) WHERE c.chunk_id STARTS WITH 'test_' RETURN count(c) AS n").single()["n"])
    print("전체 Chunk:", s.run("MATCH (c:Chunk) RETURN count(c) AS n").single()["n"])
d.close()

PLTest: 0
test_ ID: 0


전체 Chunk: 1536


## 결과 기록 (2026-11-05)
- 넣은 테스트 노드 31개, 삭제 후 PLTest 0, test_ ID 0, 전체 Chunk 1536 (변화 없음).
- Q05 질문: 93쪽 b04가 1위(점수 9.93). 질문 단어가 순위에 반영된다.
- 무관 질문 "MMR 2차 접종의 부작용은?": b04가 1위로 나온다. 질문어 가드와 제외어가 여전히 필요하다.
- 전파 질문 "홍역은 어떻게 전파되나요?": 결과 0개. 조사가 붙은 단어("홍역은", "전파되나요")가 맞지 않는다. 조사 처리 방식을 정해야 한다.
- 샘플 질문: 법 제11조 블록(18쪽 b01, b02)과 제12조 b01(19쪽)이 상위 10개에 없다. 샘플 경로의 인용 4개가 유지되지 않는다. k를 늘리거나 샘플 경로 처리를 정해야 한다.
- 경고: db.index.fulltext.queryNodes는 신버전에서 SEARCH로 바뀐다는 deprecation 경고가 나왔다. 수업 코드 방식을 따랐고 결과에는 영향이 없다.